In [4]:
import os, random

base = "/Users/ryankrishna/Patient_CTs"
all_items = sorted(os.listdir(base))
# Keep only directories named “PatientXXX” (i.e., ignore files like .DS_Store)
patients = [p for p in all_items if os.path.isdir(os.path.join(base, p)) and p.startswith("Patient")]

random.shuffle(patients)
n_train = int(len(patients) * 0.8)  # 16 train, 4 val
train_ids = patients[:n_train]
val_ids   = patients[n_train:]

with open(os.path.join(base, "train.txt"), "w") as f:
    f.write("\n".join(train_ids))
with open(os.path.join(base, "val.txt"), "w") as f:
    f.write("\n".join(val_ids))

print("TRAIN:", train_ids)
print("VAL:  ", val_ids)

TRAIN: ['Patient008', 'Patient005', 'Patient014', 'Patient007', 'Patient020', 'Patient017', 'Patient004', 'Patient006', 'Patient011', 'Patient016', 'Patient013', 'Patient019', 'Patient010', 'Patient012', 'Patient018', 'Patient009']
VAL:   ['Patient002', 'Patient001', 'Patient015', 'Patient003']


In [16]:
# Step 5 (revised): Ensure all patches are exactly the same size using SpatialPadd

import os
from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, SpatialPadd,
    RandSpatialCropd, ToTensord, Compose
)
from monai.data import Dataset, DataLoader, pad_list_data_collate

def get_transforms(patch_size=(64,64,32)):
    train_transform = Compose([
        LoadImaged(keys=["image","mask"]),
        EnsureChannelFirstd(keys=["image","mask"]),
        Orientationd(keys=["image","mask"], axcodes="RAS"),
        Spacingd(keys=["image","mask"], pixdim=(1.0,1.0,1.0), mode=("bilinear","nearest")),
        ScaleIntensityRanged(keys=["image"], a_min=-300, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        CropForegroundd(keys=["image","mask"], source_key="mask", margin=10),
        # Pad both image and mask so they are at least patch_size
        SpatialPadd(keys=["image","mask"], spatial_size=patch_size, mode="constant", constant_values=0),
        RandSpatialCropd(keys=["image","mask"], roi_size=patch_size, random_size=False),
        ToTensord(keys=["image","mask"])
    ])
    val_transform = Compose([
        LoadImaged(keys=["image","mask"]),
        EnsureChannelFirstd(keys=["image","mask"]),
        Orientationd(keys=["image","mask"], axcodes="RAS"),
        Spacingd(keys=["image","mask"], pixdim=(1.0,1.0,1.0), mode=("bilinear","nearest")),
        ScaleIntensityRanged(keys=["image"], a_min=-300, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        CropForegroundd(keys=["image","mask"], source_key="mask", margin=10),
        SpatialPadd(keys=["image","mask"], spatial_size=patch_size, mode="constant", constant_values=0),
        RandSpatialCropd(keys=["image","mask"], roi_size=patch_size, random_size=False),
        ToTensord(keys=["image","mask"])
    ])
    return train_transform, val_transform

def make_dataloaders(base_dir, batch_size=2):
    train_ids = open(os.path.join(base_dir, "train.txt")).read().splitlines()
    val_ids   = open(os.path.join(base_dir, "val.txt")).read().splitlines()

    train_files = [
        {"image": os.path.join(base_dir, pid, "ct.nii.gz"),
         "mask":  os.path.join(base_dir, pid, "pubic_mask.nii.gz")}
        for pid in train_ids
    ]
    val_files = [
        {"image": os.path.join(base_dir, pid, "ct.nii.gz"),
         "mask":  os.path.join(base_dir, pid, "pubic_mask.nii.gz")}
        for pid in val_ids
    ]

    train_t, val_t = get_transforms()
    train_ds = Dataset(data=train_files, transform=train_t)
    val_ds   = Dataset(data=val_files,   transform=val_t)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True,
        collate_fn=pad_list_data_collate
    )
    val_loader = DataLoader(
        val_ds, batch_size=1, shuffle=False,
        num_workers=1, pin_memory=True,
        collate_fn=pad_list_data_collate
    )

    return train_loader, val_loader

base = "/Users/ryankrishna/Patient_CTs"
train_loader, val_loader = make_dataloaders(base)
print("Train batches:", len(train_loader), " Val batches:", len(val_loader))

Train batches: 8  Val batches: 4


In [17]:
# Step 3 (corrected): Define and initialize the 3D U-Net model, loss function, and optimizer

import torch
from monai.networks.nets import UNet
import monai

device = "cuda" if torch.cuda.is_available() else "cpu"

# Define the 3D U-Net (use 'spatial_dims')
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm="batch"
).to(device)

# Use MONAI DiceLoss and PyTorch BCEWithLogitsLoss
dice_loss = monai.losses.DiceLoss(sigmoid=True)
bce_loss = torch.nn.BCEWithLogitsLoss()
def loss_fn(pred, mask):
    return dice_loss(pred, mask) + bce_loss(pred, mask)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Using device:", device)

Using device: cpu


In [19]:
# Step 5 (continued): Update the validation Dice calculation without compute_meandice

import torch

best_val_dice = 0.0

for epoch in range(1, 101):
    model.train()
    train_losses = []
    # Training
    for batch in tqdm(train_loader, desc=f"Epoch {epoch} [Training]"):
        img = batch["image"].to(device)
        msk = batch["mask"].to(device)
        optimizer.zero_grad()
        out = model(img)
        loss = loss_fn(out, msk)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # Validation
    model.eval()
    val_dices = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch} [Validation]"):
            img = batch["image"].to(device)
            msk = batch["mask"].to(device)
            logits = model(img)
            probs = torch.sigmoid(logits)
            pred = (probs > 0.5).float()

            # Manual Dice computation (exclude background automatically)
            intersection = torch.sum(pred * msk)
            union = torch.sum(pred) + torch.sum(msk)
            dice = (2.0 * intersection + 1e-7) / (union + 1e-7)
            val_dices.append(dice.item())

    avg_train_loss = sum(train_losses) / len(train_losses)
    avg_val_dice   = sum(val_dices)   / len(val_dices)
    print(f"Epoch {epoch:03d} | TrainLoss={avg_train_loss:.4f} | ValDice={avg_val_dice:.4f}")

    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        torch.save(model.state_dict(), "best_symphysis_unet.pth")
        print(f"  → New best model saved (ValDice={avg_val_dice:.4f})")

Epoch 1 [Validation]: 100%|█████████████████████| 4/4 [00:21<00:00,  5.42s/it]


Epoch 001 | TrainLoss=1.6868 | ValDice=0.0375
  → New best model saved (ValDice=0.0375)


Epoch 2 [Validation]: 100%|█████████████████████| 4/4 [00:21<00:00,  5.46s/it]


Epoch 002 | TrainLoss=1.6673 | ValDice=0.0378
  → New best model saved (ValDice=0.0378)


Epoch 3 [Validation]: 100%|█████████████████████| 4/4 [00:21<00:00,  5.40s/it]


...

Epoch 098 | TrainLoss=1.1249 | ValDice=0.7701


Epoch 99 [Validation]: 100%|████████████████████| 4/4 [00:22<00:00,  5.50s/it]


Epoch 099 | TrainLoss=1.1210 | ValDice=0.7784
  → New best model saved (ValDice=0.7784)


Epoch 100 [Validation]: 100%|███████████████████| 4/4 [00:21<00:00,  5.48s/it]

Epoch 100 | TrainLoss=1.1169 | ValDice=0.7764
